# Ders 2: Durağan Süreçler
- 2.1 Temel Özellikler
- 2.2 Doğrusal Süreçler
- 2.3 ARMA Süreçlerine Giriş
- 2.4 Örneklem Ortalama ve ACF Özellikleri
- 2.5 Durağan Zaman Serilerinin Öngörüsü (Durbin-Levinson, İnovasyon Algoritmaları)
- 2.6 Wold Ayrışımı


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf
from scipy.linalg import toeplitz
import warnings; warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 4)
print("Hazır!")


## 2.1 Temel Özellikler

Bir $\{X_t, t \in \mathbb{Z}\}$ süreci **durağan** (weakly stationary) eğer:
1. $E[X_t^2] < \infty$
2. $E[X_t] = \mu$ (tüm $t$ için sabit)
3. $\gamma_X(t+h, t) = \text{Cov}(X_{t+h}, X_t) = \gamma_X(h)$ (yalnızca $h$'ye bağlı)

**Otokovaryans Fonksiyonu (ACVF):** $\gamma(h)$

**Otokorelasyon Fonksiyonu (ACF):** $\rho(h) = \gamma(h)/\gamma(0)$

Önemli özellikler: $|\rho(h)| \leq 1$, $\gamma(-h) = \gamma(h)$ (simetri)


In [ ]:
# ── ACVF ve ACF: MA(1) örneği ──
# MA(1): X_t = Z_t + θ Z_{t-1}
# γ(0) = (1+θ²)σ², γ(1) = θσ², γ(h)=0 for |h|>1

sigma2 = 1.0
thetas = [0.5, -0.5, 0.9, -0.9]
lags = np.arange(0, 6)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'crimson', 'green', 'orange']

for theta, color in zip(thetas, colors):
    acvf = np.zeros(len(lags))
    acvf[0] = (1 + theta**2) * sigma2
    acvf[1] = theta * sigma2
    acf_vals = acvf / acvf[0]
    
    axes[0].stem(lags, acvf, linefmt=color, markerfmt=f'o', 
                 basefmt='k-', label=f'θ={theta}')
    axes[1].stem(lags, acf_vals, linefmt=color, markerfmt=f'o',
                 basefmt='k-', label=f'θ={theta}')

for ax, title in zip(axes, ['ACVF γ(h)', 'ACF ρ(h)']):
    ax.set_title(f'MA(1): {title}', fontsize=13)
    ax.set_xlabel('Gecikme h')
    ax.legend()
    ax.axhline(0, color='k', lw=0.5)

plt.tight_layout(); plt.show()
print("MA(1) için: ρ(0)=1, ρ(1)=θ/(1+θ²), ρ(h)=0 for |h|>1")
for theta in thetas:
    print(f"  θ={theta:5.1f} → ρ(1) = {theta/(1+theta**2):.4f}")


## 2.2 Doğrusal Süreçler

Bir **doğrusal süreç** şöyle tanımlanır:
$$X_t = \sum_{j=-\infty}^{\infty} \psi_j Z_{t-j}, \quad \{Z_t\} \sim WN(0, \sigma^2)$$

$\sum |\psi_j| < \infty$ koşulu sağlandığında süreç durağandır.

ACVF: $\gamma(h) = \sigma^2 \sum_{j=-\infty}^{\infty} \psi_{j+h} \psi_j$

**Nedensellik (Causality):** $\psi_j = 0$ for $j < 0$ ise süreç nedenseldir (gelecekteki gürültüden bağımsız).


In [ ]:
# ── Doğrusal Süreç: MA(∞) yaklaşımı ──
np.random.seed(42)
n = 500
Z = np.random.normal(0, 1, n + 100)

# Farklı ψ ağırlıkları
psi_configs = {
    'MA(3)': [1, 0.5, 0.25, 0.125],
    'Üstel Azalan': [0.8**j for j in range(20)],
    'Mevsimsel': [1 if j % 12 == 0 else 0 for j in range(25)],
}

fig, axes = plt.subplots(len(psi_configs), 2, figsize=(14, 10))

for idx, (name, psi) in enumerate(psi_configs.items()):
    psi = np.array(psi)
    q = len(psi) - 1
    
    # Seri oluştur
    X = np.array([sum(psi[j] * Z[t + 100 - j] for j in range(len(psi))) for t in range(n)])
    
    axes[idx, 0].plot(X[:150], lw=1, color='steelblue')
    axes[idx, 0].set_title(f'{name} — Zaman Serisi (ilk 150 obs)')
    axes[idx, 0].axhline(0, color='gray', ls='--', lw=0.5)
    
    # Teorik ACVF
    max_lag = 30
    gamma = np.array([sum(psi[j] * psi[j+h] for j in range(len(psi)-h)) 
                      for h in range(min(max_lag, len(psi)))])
    gamma = np.concatenate([gamma, np.zeros(max_lag - len(gamma))])
    
    axes[idx, 1].stem(range(max_lag), gamma/gamma[0], 
                       linefmt='steelblue', markerfmt='o', basefmt='k-')
    axes[idx, 1].set_title(f'{name} — Teorik ACF')
    axes[idx, 1].axhline(0, color='k', lw=0.5)

plt.tight_layout(); plt.show()


## 2.3 ARMA Süreçlerine Giriş

**ARMA(p, q) Modeli:**
$$\phi(B)X_t = \theta(B)Z_t$$

$$X_t - \phi_1 X_{t-1} - \cdots - \phi_p X_{t-p} = Z_t + \theta_1 Z_{t-1} + \cdots + \theta_q Z_{t-q}$$

Burada $B$ geri kaydırma operatörüdür: $BX_t = X_{t-1}$

- **Nedensellik:** $\phi(z) \neq 0$ for $|z| \leq 1$
- **Tersinirlik:** $\theta(z) \neq 0$ for $|z| \leq 1$


In [ ]:
# ── ARMA Simülasyonu ve Karakterizasyonu ──
from statsmodels.tsa.arima_process import arma_generate_sample

np.random.seed(1)
n = 300

models = {
    'AR(1) φ=0.8':        {'ar': [1, -0.8],  'ma': [1]},
    'AR(2) φ₁=0.5,φ₂=0.3': {'ar': [1,-0.5,-0.3], 'ma': [1]},
    'MA(2) θ₁=0.7,θ₂=-0.4': {'ar': [1],     'ma': [1, 0.7, -0.4]},
    'ARMA(1,1) φ=0.6,θ=0.4': {'ar': [1,-0.6], 'ma': [1, 0.4]},
}

fig, axes = plt.subplots(len(models), 3, figsize=(16, 12))

for i, (name, params) in enumerate(models.items()):
    X = arma_generate_sample(params['ar'], params['ma'], n, scale=1)
    
    axes[i, 0].plot(X, lw=0.8, color='steelblue')
    axes[i, 0].set_title(f'{name}')
    axes[i, 0].axhline(0, color='gray', ls='--', lw=0.5)
    
    plot_acf(X, lags=20, ax=axes[i, 1], color='steelblue', title='ACF')
    plot_pacf(X, lags=20, ax=axes[i, 2], color='darkorange', title='PACF')

plt.tight_layout(); plt.show()

print("Model Tanıma Rehberi:")
print("  ACF kesiliyor, PACF azalıyor → AR(p)  modeli")
print("  ACF azalıyor,  PACF kesiliyor → MA(q)  modeli")
print("  Her ikisi de azalıyor         → ARMA(p,q) modeli")


## 2.5 Durağan Zaman Serilerinin Öngörüsü

### 2.5.3 Durbin-Levinson Algoritması

PACF $\{\phi_{nn}\}$'yi rekursif olarak hesaplar:

$$\phi_{n+1,n+1} = \frac{\rho(n+1) - \sum_{j=1}^{n}\phi_{nj}\rho(n+1-j)}{1 - \sum_{j=1}^{n}\phi_{nj}\rho(j)}$$

Bu, $n$-adım öngörüsünde en iyi doğrusal tahminciyi verir:
$$\hat{X}_{n+1} = \phi_{n1}X_n + \phi_{n2}X_{n-1} + \cdots + \phi_{nn}X_1$$


In [ ]:
# ── Durbin-Levinson Algoritması (elle uygulama) ──

def durbin_levinson(rho, n_steps):
    '''
    rho: ACF değerleri [ρ(0), ρ(1), ρ(2), ...]
    Döndürür: PACF değerleri φ_{nn}
    '''
    phi = {}   # phi[n][j] = φ_{n,j}
    v = {}     # tahmin hatası varyansları
    pacf_vals = []
    
    phi[1] = {1: rho[1]}
    v[1] = 1 - rho[1]**2
    pacf_vals.append(rho[1])
    
    for n in range(1, n_steps):
        num = rho[n+1] - sum(phi[n].get(j,0) * rho[n+1-j] for j in range(1, n+1))
        denom = v[n]
        phi_new = num / denom
        pacf_vals.append(phi_new)
        
        phi[n+1] = {}
        phi[n+1][n+1] = phi_new
        for j in range(1, n+1):
            phi[n+1][j] = phi[n].get(j,0) - phi_new * phi[n].get(n+1-j, 0)
        
        v[n+1] = v[n] * (1 - phi_new**2)
    
    return np.array(pacf_vals), v

# AR(2) sürecinin teorik ACF'si
phi1, phi2, sigma2 = 0.7, -0.3, 1.0
# Yule-Walker'dan teorik ACF:
rho1 = phi1 / (1 - phi2)
rho2 = phi2 + phi1 * rho1
rho_vals = [1.0, rho1, rho2]
for h in range(3, 25):
    rho_vals.append(phi1 * rho_vals[-1] + phi2 * rho_vals[-2])

pacf_dl, variances = durbin_levinson(rho_vals, 15)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].stem(range(len(rho_vals)), rho_vals, linefmt='steelblue', 
             markerfmt='o', basefmt='k-')
axes[0].set_title('Teorik ACF — AR(2) φ₁=0.7, φ₂=-0.3')
axes[0].set_xlabel('Gecikme h')

axes[1].stem(range(1, len(pacf_dl)+1), pacf_dl, linefmt='darkorange', 
             markerfmt='o', basefmt='k-')
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_title('Durbin-Levinson PACF')
axes[1].set_xlabel('Gecikme n')
axes[1].axhline(phi1, color='red', ls='--', alpha=0.5, label=f'φ₁={phi1}')
axes[1].axhline(phi2, color='blue', ls='--', alpha=0.5, label=f'φ₂={phi2}')
axes[1].legend()

axes[2].plot(list(variances.values()), 'g-o', markersize=5)
axes[2].set_title('Tahmin Hatası Varyansı $v_n$')
axes[2].set_xlabel('n')
axes[2].axhline(sigma2*(1-phi1**2-phi2**2-phi1*phi2*rho1), color='red', 
                ls='--', label='Teorik minimum')
axes[2].legend()

plt.tight_layout(); plt.show()
print(f"PACF φ₁₁ = {pacf_dl[0]:.4f} (Beklenen: {rho1:.4f})")
print(f"PACF φ₂₂ = {pacf_dl[1]:.4f} (Beklenen: {phi2:.4f})")
print(f"PACF φ₃₃ = {pacf_dl[2]:.6f} (Beklenen: ≈ 0)")


## 2.6 Wold Ayrışımı

**Teorem (Wold, 1938):** Her sıfır-ortalıklı durağan süreç $\{X_t\}$, şöyle yazılabilir:
$$X_t = \sum_{j=0}^{\infty} \psi_j Z_{t-j} + V_t$$

Burada:
- $\{Z_t\} \sim WN(0, \sigma^2)$: inovasyon süreci
- $\{V_t\}$: deterministik süreç ($Z_t$ ile ilişkisiz)
- $\sum \psi_j^2 < \infty$, $\psi_0 = 1$

Bu teoremi AR(1) örneğiyle anlayalım:


In [ ]:
# ── Wold Ayrışımı: AR(1) → MA(∞) ──

phi = 0.7
max_terms = [3, 10, 50, 200]

np.random.seed(42)
Z = np.random.normal(0, 1, 1000)
n_show = 100

fig, axes = plt.subplots(len(max_terms)+1, 1, figsize=(13, 12), sharex=True)

# Gerçek AR(1)
X_true = np.zeros(1000)
for t in range(1, 1000):
    X_true[t] = phi * X_true[t-1] + Z[t]

axes[0].plot(X_true[900:], 'k', lw=1.5, label='Gerçek AR(1)')
axes[0].set_title('AR(1): $X_t = 0.7 X_{t-1} + Z_t$')
axes[0].legend(); axes[0].axhline(0, color='gray', ls='--', lw=0.5)

# Wold MA(∞) yaklaşımı
for idx, q in enumerate(max_terms):
    psi = np.array([phi**j for j in range(q)])
    X_approx = np.array([sum(psi[j] * Z[t-j] for j in range(min(q, t+1))) 
                         for t in range(900, 1000)])
    axes[idx+1].plot(X_approx, color='steelblue', lw=1, alpha=0.8, label=f'MA(∞) q={q}')
    axes[idx+1].plot(X_true[900:], 'k--', lw=0.8, alpha=0.4, label='Gerçek')
    axes[idx+1].set_title(f'Wold Yaklaşımı: q={q} terim, ψⱼ = 0.7ʲ')
    axes[idx+1].legend(fontsize=9)
    axes[idx+1].axhline(0, color='gray', ls='--', lw=0.5)
    
    mse = np.mean((X_approx - X_true[900:])**2)
    print(f"q={q:4d} terim: MSE = {mse:.6f}")

axes[-1].set_xlabel('Zaman')
plt.tight_layout(); plt.show()
print("\nq artıkça MA(∞) yaklaşımı AR(1)'e daha çok benziyor!")


## Özet

| Kavram | Formül | Önem |
|--------|--------|------|
| **Durağanlık** | $\gamma(t+h,t)=\gamma(h)$ | Temel varsayım |
| **ACVF** | $\gamma(h) = \text{Cov}(X_{t+h}, X_t)$ | Kovaryans yapısı |
| **ACF** | $\rho(h) = \gamma(h)/\gamma(0)$ | Korelasyon yapısı |
| **PACF** | $\phi_{nn}$ | Doğrudan etki |
| **Durbin-Levinson** | Rekursif $\phi_{nn}$ hesabı | Verimli öngörü |
| **Wold Ayrışımı** | $X_t = \sum \psi_j Z_{t-j} + V_t$ | ARMA modellemesinin temeli |

